# Finding MIN and MAX with a Two-Layer Neural Network

In this notebook, you will **manually set the weights and biases** of a two-layer neural network  
so that it correctly identifies the **minimum** and **maximum** element in a 3-element input vector.

There is no training — you need to design the weights by reasoning about what each layer should compute.

---
**Goal:** Edit `weight1`, `bias1`, `weight2`, `bias2` (separately for MIN and MAX)  
so that all 8 test cases pass (`PASSED`).

> ⚠️ **Important:** Every time you edit the code cell, you **must run it** to save your changes.  
> Then run the next cell to execute the file and check the results.


---
## Step 1: Understand the Network

The network (`TwoLayers`) has the following structure:

```
Input x = (x0, x1, x2)  — three random numbers in [0, 1]
         ↓
  Layer 1 (fc1):  h = sign(x @ W1 + b1)     — hidden layer (3 neurons)
         ↓
  Layer 2 (fc2):  y = sign(h @ W2 + b2)     — output layer (3 neurons)
         ↓
Output y = (y0, y1, y2)  — one-hot ±1 vector
```

**Output encoding:**
- The output is a vector of three values, each +1 or -1.
- Exactly **one** element is **+1** (the position of the min or max).
- The other two elements are **-1**.

Example (find MIN, x1 is smallest):
```
Input:  x = [0.8, 0.1, 0.5]
Target: y = [-1, +1, -1]
```

### About the weight matrices

In this code, the weight matrices are stored in **column-major** convention:  
Column `j` of `weight1` contains the weights that feed into hidden neuron `j`.

So the computation is:
$$h_j = \text{sign}\left(\sum_i x_i \cdot \text{weight1}[i, j] + \text{bias1}[j]\right)$$

Similarly for Layer 2:
$$y_j = \text{sign}\left(\sum_i h_i \cdot \text{weight2}[i, j] + \text{bias2}[j]\right)$$


---
## Step 2: Design Layer 1 — What Should the Hidden Neurons Compute?

You have 3 inputs and 3 hidden neurons. The hidden layer uses `sign` activation,  
so each hidden neuron can only output **+1 or -1**.

**Key insight:** A neuron can compare two values!

Consider the computation:
$$h = \text{sign}(x_i - x_j)$$
- If $x_i > x_j$: $h = +1$
- If $x_i < x_j$: $h = -1$

With 3 inputs, there are **3 pairs** to compare:
- $x_0$ vs $x_1$
- $x_0$ vs $x_2$
- $x_1$ vs $x_2$

💡 **Think about it:**  
If hidden neuron 0 computes `sign(x0 - x1)`, what values of `weight1[:, 0]` and `bias1[0]` achieve this?

Try to design all 3 hidden neurons this way, then move to Layer 2.


---
## Step 3: Design Layer 2 — How to Identify MIN or MAX from the Comparisons?

Suppose Layer 1 computes the three pairwise comparisons: `h0 = sign(x0-x1)`, `h1 = sign(x0-x2)`, `h2 = sign(x1-x2)`.

**For MIN:** Think about when each element is the minimum.

Fill in the table below (assuming no ties):

| Minimum element | h0 = sign(x0−x1) | h1 = sign(x0−x2) | h2 = sign(x1−x2) |
|:---:|:---:|:---:|:---:|
| x0 is min | ? | ? | ± (either) |
| x1 is min | ? | ± (either) | ? |
| x2 is min | ± (either) | ? | ? |

> Fill in the **?** values yourself. What must h0 and h1 be when x0 is the minimum?

Once you know which combination of hidden values corresponds to each case,  
design output neuron `y_k` so that:
- `y_k = +1` when `x_k` is the minimum
- `y_k = -1` otherwise

This means writing a linear inequality over `h0`, `h1`, `h2` (plus bias) that is positive only in the right case.

**For MAX:** The same approach applies — just think about when each element is the *largest*.

💡 **Tip:** You only need **two** of the three hidden values to determine each output.  
Which two are sufficient for each output neuron? (The third can have coefficient 0.)


---
## Step 4: Systematic Approach — Solving the Inequalities

For each output neuron `y_k`, you need:
$$\text{sign}(a \cdot h_0 + b \cdot h_1 + c \cdot h_2 + d) = \begin{cases} +1 & \text{if } x_k \text{ is min/max} \\ -1 & \text{otherwise} \end{cases}$$

Since `h` values are ±1, substituting all relevant combinations gives you a set of **linear inequalities** in `a, b, c, d`:
- For the case where `y_k` should be +1: `a·h0 + b·h1 + c·h2 + d > 0`
- For all other cases where `y_k` should be -1: `a·h0 + b·h1 + c·h2 + d < 0`

Try simple values like `{-1, 0, +1}` for `a, b, c` and small integers for `d`.

**Reminder:** There are multiple valid solutions — any values that satisfy all the inequalities will work.


In [ ]:
# Mount Google Drive and set up project folder
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH="/content/drive/MyDrive/Practical-ML/logic"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls *.py

---
## How to Edit and Test Your Answer

Follow these steps every time you try new values:

1. **Edit** the `weight1`, `bias1`, `weight2`, `bias2` values in the `%%writefile` cell below.
2. **Run the `%%writefile` cell** — this saves your updated code as `minmax.py` on Google Drive.  
   *(Skipping this step means your changes are NOT saved — the old file will run instead.)*
3. **Run the next cell** (`!python minmax.py`) to execute the file and see the results.

> 🔁 **Repeat steps 1 → 2 → 3 every time you change the values.**


In [ ]:
# Execute this cell after editing, then run the next cell to check the results.
%%writefile minmax.py
import torch
import torch.nn as nn
import torch.nn.functional as F

# Network definition: two linear layers each followed by sign activation
class TwoLayers(nn.Module):
    def __init__(self):
        super(TwoLayers, self).__init__()
        # Layer 1: 3 inputs -> 3 hidden neurons
        self.fc1 = nn.Linear(3, 3)
        # Layer 2: 3 hidden neurons -> 3 output neurons
        self.fc2 = nn.Linear(3, 3)

    def forward(self, x):
        x = self.fc1(x)       # hidden: h = W1^T * x + b1  (PyTorch convention)
        x = torch.sign(x)     # sign activation
        x = self.fc2(x)       # output: y = W2^T * h + b2
        x = torch.sign(x)     # sign activation
        return x

    def set_weight_bias(self, w1, b1, w2, b2):
        # NOTE: weight matrices are stored transposed here.
        # weight1[i, j] = weight from input i to hidden neuron j.
        # The computation is: h = sign(X @ weight1 + bias1)
        self.fc1.weight.data = w1.transpose(0, 1).clone()
        self.fc1.bias.data   = b1.clone()
        self.fc2.weight.data = w2.transpose(0, 1).clone()
        self.fc2.bias.data   = b2.clone()

# 4 random 3-element input vectors in [0, 1]
N = 4
X = torch.rand(N, 3)
net = TwoLayers()

crr = 0

# ----------------------------------------------------------------
### find MIN ###
# TODO: Set weight1, bias1, weight2, bias2 so that the network
#       outputs a one-hot ±1 vector indicating the MINIMUM element.
#
# Output format:
#   y = [+1, -1, -1]  if x[0] is the smallest
#   y = [-1, +1, -1]  if x[1] is the smallest
#   y = [-1, -1, +1]  if x[2] is the smallest
#
# Hint: Think about what each hidden neuron should compare.
#       Then design the output layer to detect which element is smallest.
weight1 = [[1, 0, 0],   # <- Edit this 3x3 matrix
           [0, 1, 0],
           [0, 0, 1]]
bias1   = [0, 0, 0]     # <- Edit this

weight2 = [[1, 0, 0],   # <- Edit this 3x3 matrix
           [0, 1, 0],
           [0, 0, 1]]
bias2   = [0, 0, 0]     # <- Edit this
# ----------------------------------------------------------------

print("find MIN:")
net.set_weight_bias(
    torch.tensor(weight1, dtype=torch.float32),
    torch.tensor(bias1,   dtype=torch.float32),
    torch.tensor(weight2, dtype=torch.float32),
    torch.tensor(bias2,   dtype=torch.float32)
)
Y = net(X)
Y = Y.to(dtype=torch.int32)
for i in range(N):
    t = -torch.ones(3, dtype=torch.int32)
    ind = X[i, :].argmin()
    t[ind] = +1
    print("{:.3f} {:.3f} {:.3f}".format(X[i,0], X[i,1], X[i,2]), end=" -> ")
    print("{:+d} {:+d} {:+d}".format(Y[i,0], Y[i,1], Y[i,2]), end=" ")
    print("[{:+d} {:+d} {:+d}]".format(t[0], t[1], t[2]), end=" ")
    if (Y[i, :] == t).all(): print("OK"); crr += 1
    else: print("NG")
print()


# ----------------------------------------------------------------
### find MAX ###
# TODO: Set weight1, bias1, weight2, bias2 so that the network
#       outputs a one-hot ±1 vector indicating the MAXIMUM element.
#
# Output format:
#   y = [+1, -1, -1]  if x[0] is the largest
#   y = [-1, +1, -1]  if x[1] is the largest
#   y = [-1, -1, +1]  if x[2] is the largest
#
# Hint: The same Layer 1 comparison idea applies.
#       Think about how the signs of the comparisons differ for MAX vs MIN.
weight1 = [[1, 0, 0],   # <- Edit this 3x3 matrix
           [0, 1, 0],
           [0, 0, 1]]
bias1   = [0, 0, 0]     # <- Edit this

weight2 = [[1, 0, 0],   # <- Edit this 3x3 matrix
           [0, 1, 0],
           [0, 0, 1]]
bias2   = [0, 0, 0]     # <- Edit this
# ----------------------------------------------------------------

print("find MAX:")
net.set_weight_bias(
    torch.tensor(weight1, dtype=torch.float32),
    torch.tensor(bias1,   dtype=torch.float32),
    torch.tensor(weight2, dtype=torch.float32),
    torch.tensor(bias2,   dtype=torch.float32)
)
Y = net(X)
Y = Y.to(dtype=torch.int32)
for i in range(N):
    t = -torch.ones(3, dtype=torch.int32)
    ind = X[i, :].argmax()
    t[ind] = +1
    print("{:.3f} {:.3f} {:.3f}".format(X[i,0], X[i,1], X[i,2]), end=" -> ")
    print("{:+d} {:+d} {:+d}".format(Y[i,0], Y[i,1], Y[i,2]), end=" ")
    print("[{:+d} {:+d} {:+d}]".format(t[0], t[1], t[2]), end=" ")
    if (Y[i, :] == t).all(): print("OK"); crr += 1
    else: print("NG")
print()

if crr >= 8: print("PASSED")
else: print("FAILED")


In [ ]:
# Run the saved file to check your answers
%cd "{PROJECT_PATH}"
!python minmax.py

---
## Example format for submission

```
MIN:
  weight1 = [[?, ?, ?],    bias1 = [?, ?, ?]
              [?, ?, ?],
              [?, ?, ?]]
  weight2 = [[?, ?, ?],    bias2 = [?, ?, ?]
              [?, ?, ?],
              [?, ?, ?]]

MAX:
  (same format)
```
